<a href="https://colab.research.google.com/github/25bca089-Razi/smart-nlp-sentiment-analyzer/blob/main/AI_ML_prj_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# INSTALLATION
# ============================================================

!pip install -q gradio textblob nltk pandas


# ============================================================
# IMPORT LIBRARIES
# ============================================================

import gradio as gr
import nltk
import re
import pandas as pd

from textblob import TextBlob
from collections import Counter
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag


# ============================================================
# DOWNLOAD REQUIRED DATASETS
# ============================================================

for package in [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng"
]:
    nltk.download(package, quiet=True)


# ============================================================
# INITIALIZE NLP COMPONENTS
# ============================================================

word_stemmer = PorterStemmer()
word_lemmatizer = WordNetLemmatizer()

ENGLISH_STOPWORDS = set(stopwords.words("english"))


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_sentence(sentence):

    sentence = sentence.lower()

    sentence = re.sub(
        r'https?://\S+|www\.\S+',
        '',
        sentence
    )

    sentence = re.sub(
        r'[^a-zA-Z\s]',
        '',
        sentence
    )

    sentence = re.sub(
        r'\s+',
        ' ',
        sentence
    )

    return sentence.strip()


# ============================================================
# POS TAG CONVERSION
# ============================================================

def identify_word_type(tag):

    if tag.startswith('J'):
        return wordnet.ADJ

    if tag.startswith('V'):
        return wordnet.VERB

    if tag.startswith('N'):
        return wordnet.NOUN

    if tag.startswith('R'):
        return wordnet.ADV

    return wordnet.NOUN


# ============================================================
# TEXT PROCESSING PIPELINE
# ============================================================

def process_sentence(sentence):

    normalized = normalize_sentence(sentence)

    word_list = word_tokenize(normalized)

    useful_words = [
        word for word in word_list
        if word not in ENGLISH_STOPWORDS
    ]

    stemmed = [
        word_stemmer.stem(word)
        for word in useful_words
    ]

    tagged = pos_tag(useful_words)

    lemmatized = [
        word_lemmatizer.lemmatize(
            word,
            identify_word_type(tag)
        )
        for word, tag in tagged
    ]

    return normalized, useful_words, stemmed, lemmatized


# ============================================================
# SENTIMENT CLASSIFICATION
# ============================================================

def classify_emotion(score):

    if score > 0.1:
        return "😊 Positive"

    elif score < -0.1:
        return "😞 Negative"

    return "😐 Neutral"


# ============================================================
# MAIN ANALYSIS FUNCTION
# ============================================================

def generate_report(user_text):

    if not user_text or not user_text.strip():

        return (
            "Please provide some text.",
            "",
            "",
            "",
            "",
            "",
            ""
        )

    # Sentiment calculation
    text_analysis = TextBlob(user_text)

    polarity = text_analysis.sentiment.polarity
    subjectivity = text_analysis.sentiment.subjectivity

    emotion = classify_emotion(polarity)

    # NLP processing
    normalized, tokens, stems, lemmas = process_sentence(
        user_text
    )

    # Word frequency calculation
    word_counts = Counter(lemmas)

    common_words = word_counts.most_common(10)

    if common_words:

        frequency_report = "\n".join(
            [
                f"{index}. {word} : {count} occurrence(s)"
                for index, (word, count)
                in enumerate(common_words, 1)
            ]
        )

    else:

        frequency_report = "No meaningful words detected."

    # Generate statistics
    total_words = len(user_text.split())

    unique_words = len(set(tokens))

    total_characters = len(user_text)

    total_sentences = len(text_analysis.sentences)

    report = f"""
TEXT ANALYTICS REPORT
----------------------------

Total Characters : {total_characters}
Total Words      : {total_words}
Filtered Tokens  : {len(tokens)}
Unique Words     : {unique_words}
Total Sentences  : {total_sentences}

SENTIMENT DETAILS
----------------------------

Polarity Score    : {polarity:.3f}
Subjectivity Score: {subjectivity:.3f}

Polarity Range:
-1.0 = Negative
 0.0 = Neutral
+1.0 = Positive

Subjectivity Range:
0.0 = Objective
1.0 = Subjective

TOP FREQUENT WORDS
----------------------------
{frequency_report}
"""

    return (
        emotion,
        normalized,
        " | ".join(tokens),
        " | ".join(stems),
        " | ".join(lemmas),
        f"{polarity:.3f} / 1.0",
        report
    )


# ============================================================
# RESET FUNCTION
# ============================================================

def reset_dashboard():

    return (
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    )


# ============================================================
# GRADIO DASHBOARD DESIGN
# ============================================================

custom_css = """
.gradio-container {
    max-width: 1100px !important;
    margin: auto !important;
}

#main-title {
    text-align: center;
}

#main-title h1 {
    color: #2563eb;
    font-size: 32px;
}

#analyze-btn {
    background: #2563eb;
    color: white;
}
"""


with gr.Blocks(
    theme=gr.themes.Glass(
        primary_hue="blue",
        secondary_hue="indigo"
    ),
    css=custom_css,
    title="Smart NLP Text Analyzer"
) as dashboard:

    # HEADER
    gr.Markdown(
        """
        # 🔍 Smart NLP Text Analyzer

        ### Intelligent Text Processing and Sentiment Detection

        Analyze written content using Natural Language Processing.
        """,
        elem_id="main-title"
    )

    gr.Markdown("---")

    # INPUT AREA
    with gr.Group():

        gr.Markdown("### 📝 Input Section")

        input_area = gr.Textbox(
            label="Enter your sentence or paragraph",
            placeholder="Type or paste your text here...",
            lines=5
        )

        with gr.Row():

            analyze_btn = gr.Button(
                "🚀 Start Analysis",
                variant="primary",
                elem_id="analyze-btn"
            )

            reset_btn = gr.Button(
                "↻ Reset",
                variant="secondary"
            )

    # SENTIMENT SECTION
    gr.Markdown("---")
    gr.Markdown("## 📊 Sentiment Detection")

    with gr.Row():

        emotion_result = gr.Textbox(
            label="Detected Sentiment",
            interactive=False
        )

        score_result = gr.Textbox(
            label="Polarity Score",
            interactive=False
        )

    # TEXT PROCESSING SECTION
    gr.Markdown("---")
    gr.Markdown("## ⚙️ Text Preprocessing Results")

    with gr.Tab("Cleaned Text"):

        clean_result = gr.Textbox(
            label="Normalized Output",
            lines=4,
            interactive=False
        )

    with gr.Tab("Tokenization"):

        token_result = gr.Textbox(
            label="Filtered Tokens",
            lines=4,
            interactive=False
        )

    with gr.Tab("Stemming"):

        stem_result = gr.Textbox(
            label="Stemmed Vocabulary",
            lines=4,
            interactive=False
        )

    with gr.Tab("Lemmatization"):

        lemma_result = gr.Textbox(
            label="Lemmatized Vocabulary",
            lines=4,
            interactive=False
        )

    # STATISTICS SECTION
    gr.Markdown("---")
    gr.Markdown("## 📈 Detailed Analysis")

    report_result = gr.Textbox(
        label="Text Analysis Report",
        lines=18,
        interactive=False
    )

    # EXAMPLE INPUTS
    gr.Markdown("---")
    gr.Markdown("## 💡 Try Sample Sentences")

    gr.Examples(
        examples=[
            ["The new smartphone has an excellent camera and amazing battery life."],
            ["I am unhappy with the poor customer service and delayed delivery."],
            ["The meeting is scheduled for tomorrow afternoon."],
            ["The food was delicious but the restaurant was overcrowded."],
        ],
        inputs=input_area
    )

    # BUTTON ACTIONS
    analyze_btn.click(
        fn=generate_report,
        inputs=input_area,
        outputs=[
            emotion_result,
            clean_result,
            token_result,
            stem_result,
            lemma_result,
            score_result,
            report_result
        ]
    )

    reset_btn.click(
        fn=reset_dashboard,
        inputs=[],
        outputs=[
            input_area,
            emotion_result,
            clean_result,
            token_result,
            stem_result,
            lemma_result,
            score_result,
            report_result
        ]
    )


# ============================================================
# START APPLICATION
# ============================================================

dashboard.launch(share=True)


/tmp/ipykernel_5807/584059781.py:292: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4ac0a7e95f231c898e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
